# Fair Market Rents — When the Number Is a Decision

Every source in this corpus so far publishes a *measurement*: a survey estimate with a
margin, a model output, an administrative count, a suppressed cell. This notebook is about a
different kind of number.

A **Fair Market Rent** is defined by regulation as an estimate:

> "Fair Market Rents, as defined in 24 CFR 888.113 are estimates of 40th percentile gross
> rents for standard quality units within a metropolitan area or nonmetropolitan county."

And it is used as a rule. FMRs set payment standards for the Housing Choice Voucher program,
renewal rents for expiring project-based Section 8 contracts, rent ceilings in HOME and the
Emergency Solutions Grants program, Continuum of Care maximums, and Public Housing flat
rents. 42 USC 1437f requires them posted at least 30 days before they take effect, at the
start of the federal fiscal year.

So the number is simultaneously an estimate of something and an instruction to somebody. The
two roles pull in different directions, and every difficulty in this notebook comes from
that. A measurement wants to be accurate about a place and a moment. A rule wants to be
stable, forecast forward, uniform across an administrative area, and appealable — and each of
those four properties moves the published number away from the thing it is named after.

**The regime.** Call it the *administratively determined* number, and note what it disables:

- It publishes **no uncertainty**, like LAUS — so no significance test, no ranking claim.
- It is **revised after it takes effect**, unlike a survey vintage — and HUD publishes no
  revision series, so the revision is visible only by differencing two files.
- Its **geography is set by statute**, not by the housing market, so it is constant across
  places that are not alike.
- And its **definition describes the input, not the output**. §1 measures how far apart
  those two are.

## Learning Objectives

1. **Distinguish** a published number's *definition* from the *quantity it is published as*,
   and measure the gap rather than assuming it away.
2. **Quantify** what a statutory geography costs: how much variation a single binding number
   is required to ignore, separated from the level shift that would otherwise mask it.
3. **Detect** that a publisher's machine-readable distribution is not the operative schedule,
   by checking it against the authoritative files rather than trusting its label.
4. **Recover** a revision that has no revision series, by differencing two publications of
   the same fiscal year.
5. **Refuse** a decomposition the source does not support, after establishing that the file
   offered for it covers a selected subset.

## Prerequisites

**Knowledge.** D27 (rent burden — ACS housing tables, denominators, bracket structure) and
§1.3 of the Data Source Handbook (revision, and why a vintage must be pinned). D09 helps:
it is the other notebook where a policy, not a sample, decides what you get to see.

**Credentials.** `CENSUS_API_KEY` and `HUD_API_KEY`. The pinned HUD files need neither; the
API cross-check in §4b needs the HUD token.

**Data.** `data/hud/fmr/` must be present. See *Data Provenance* below and that directory's
`SOURCES.md` for the digests and the retrieval constraint.

## Data Provenance

| Source | What | Regime | Vintage |
|---|---|---|---|
| HUD FMR `FY27_FMRs` | County/town FMR by bedroom, FY2027 | Administratively determined | Pinned bulk file |
| HUD FMR `FY26_FMRs` | The same for FY2026, as first published | Administratively determined | Pinned bulk file |
| HUD FMR `FY26_FMRs_revised` | FY2026 after reevaluation | Administratively determined | Pinned bulk file |
| HUD `FY27_Unadjusted` | Pre-adjustment rents, subset of areas | Administratively determined | Pinned bulk file |
| HUD ArcGIS `Fair_Market_Rents` | HUD's own machine-readable layer | **Unlabelled** — §3 establishes what it is | Live query |
| HUD USER FMR API | The same figures, per county per fiscal year | Administratively determined | Live query, year pinned |
| Census ACS 5-Year 2023, B25031 | Median gross rent by bedroom count, county | Margins of error | Dataset year |

**How these files were obtained, and a correction worth keeping.** HUD's `/portal/` paths —
where the bulk files live — answer a non-browser client with **HTTP 202, a zero-length body,
and `x-amzn-waf-action: challenge`**. That is HUD's own bot defence, and 202 is a *success*
code, so a client testing `status_code < 400` accepts it and parses nothing. During this
build that response was first misdiagnosed as an egress block. It is not: the FMR **API** on
the same host is unchallenged and works with a key, as §4b demonstrates by re-fetching these
values from HUD directly.

The bulk files remain pinned rather than fetched, for two reasons that are about the source
rather than the network: the API serves one entity per request under a 60/minute limit, and
it serves **only the current value**, so FY2026-as-first-published has no API equivalent at
all. Their digests are checked on every load.

In [1]:
"""Configuration. Everything that fixes an answer is named here, once."""
from __future__ import annotations

import json
import urllib.request
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

import sys
from pathlib import Path

#: Find the package by walking UP from wherever this runs, so the notebook works
#: whether it is executed in place or from the repository root. Sixteen notebooks
#: imported `krl_tutorials_data` with no path discovery and therefore ran only
#: from the root -- which nobody noticed while every run happened to start there.
_repo = next((c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
              if (c / "src" / "krl_tutorials_data").is_dir()), None)
if _repo is None:
    raise SystemExit("krl_tutorials_data not found above " + str(Path.cwd()))
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo / "src"))

from krl_tutorials_data.connectors import BulkFileSource, frame_from_bulk_file
from krl_tutorials_data.sources import fetch_acs_wide

#: Anchored to the repository root rather than the working directory.
#: This notebook resolved `data/hud/...` relative to cwd, so it ran only from
#: the repo root -- the same dependence that put sixteen provenance records in
#: a stray folder.
REPO_ROOT = next(c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                 if (c / "src" / "krl_tutorials_data").is_dir())

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

# The bedroom size everything is reported at. FMRs are published for 0-4 bedrooms;
# the 2-bedroom unit is HUD's own reference size and the one the ACS publishes a
# directly comparable median for.
BEDROOM = 2
FMR_COL = f"fmr_{BEDROOM}"
ACS_RENT = "B25031_004E"   # median gross rent, 2-bedroom units
ACS_RENT_MOE = "B25031_004M"

ACS_YEAR = 2023            # the latest 5-year release; pinned, not "latest"
FISCAL_YEAR = 2027         # the FMR schedule in force

DATA = REPO_ROOT / Path("data/hud/fmr")

# HUD's open-data feature layer. Carries no fiscal-year field; section 3 establishes
# which schedule it actually holds.
ARCGIS = (
    "https://services.arcgis.com/VTyQ9soqVukalItT/arcgis/rest/services/"
    "Fair_Market_Rents/FeatureServer/0/query"
)

print(f"FY{FISCAL_YEAR} FMR schedule vs ACS {ACS_YEAR} 5-year, {BEDROOM}-bedroom units")

FY2027 FMR schedule vs ACS 2023 5-year, 2-bedroom units


In [2]:
"""Load the four pinned HUD files.

Each digest was computed at retrieval and is checked on every load: a bulk publisher
overwrites downloads in place, so "the FY2027 FMR file" is not an identifier. A changed
byte fails here rather than changing a number downstream.
"""
HUD_FILES = {
    "fy27": BulkFileSource(
        path=DATA / "FY27_FMRs.csv",
        publisher="U.S. Department of Housing and Urban Development (HUD USER)",
        dataset="Fair Market Rents, FY2027, county level",
        vintage="FY2027",
        retrieved_on=date(2026, 9, 14),
        sha256="a52b9f9a6cd7e8f227c270a70acefba1ffde35dd78d5fba6813037d0db6f68dc",
        terms_url="https://www.huduser.gov/portal/about/pdr_data_license.html",
        notes=(
            "Derived from FY27_FMRs.xlsx sheet 1; trailing empty rows dropped.",
            "Upstream xlsx SHA-256 423d1c6f8ef2...d1b7, 375,708 bytes.",
        ),
    ),
    "fy26": BulkFileSource(
        path=DATA / "FY26_FMRs.csv",
        publisher="U.S. Department of Housing and Urban Development (HUD USER)",
        dataset="Fair Market Rents, FY2026, county level, as first published",
        vintage="FY2026 original",
        retrieved_on=date(2026, 9, 14),
        sha256="63bab212511808e2ef6351b9e85e46eb87b024e5c4ddcc50d7f515c4f42afa12",
        terms_url="https://www.huduser.gov/portal/about/pdr_data_license.html",
        notes=("Columns reduced to fips + fmr_0..fmr_4. Join on fips only.",),
    ),
    "fy26r": BulkFileSource(
        path=DATA / "FY26_FMRs_revised.csv",
        publisher="U.S. Department of Housing and Urban Development (HUD USER)",
        dataset="Fair Market Rents, FY2026, county level, revised",
        vintage="FY2026 revised",
        retrieved_on=date(2026, 9, 14),
        sha256="fe2162d980c506aebe6f6872ac1fbdb33448ddf66b2ec19d3342cf79fd78e5f8",
        terms_url="https://www.huduser.gov/portal/about/pdr_data_license.html",
        notes=("Columns reduced to fips + fmr_0..fmr_4. Join on fips only.",),
    ),
    "unadj": BulkFileSource(
        path=DATA / "FY27_Unadjusted.csv",
        publisher="U.S. Department of Housing and Urban Development (HUD USER)",
        dataset="FY2027 unadjusted rents (public housing flat rents)",
        vintage="FY2027",
        retrieved_on=date(2026, 9, 14),
        sha256="5b447ab9551143b83203d4a7f9a092a7c8e6f1c6aba41142d50d4e4777522d42",
        terms_url="https://www.huduser.gov/portal/about/pdr_data_license.html",
        notes=("HUD states this file is only for setting public housing flat rents.",),
    ),
}

FMR_COLS = [f"fmr_{b}" for b in range(5)]


def read_fmr(path: Path) -> pd.DataFrame:
    """Read one HUD CSV. `fips` is a 10-character code and must not become an int."""
    return pd.read_csv(path, dtype={"fips": str, "hud_area_code": str, "state": str})


frames, provenance = {}, {}
for key, src in HUD_FILES.items():
    frames[key], provenance[key] = frame_from_bulk_file(
        src, read_fmr, expected_columns={"fips": "10-char county/town code, SSCCC + 99999"}
    )
    print(f"{src.vintage:<16} {len(frames[key]):>5} rows  pin verified  {src.path.name}")

fy27, fy26, fy26r, unadj = frames["fy27"], frames["fy26"], frames["fy26r"], frames["unadj"]

FY2027            4764 rows  pin verified  FY27_FMRs.csv


FY2026 original   4764 rows  pin verified  FY26_FMRs.csv
FY2026 revised    4764 rows  pin verified  FY26_FMRs_revised.csv
FY2027            1057 rows  pin verified  FY27_Unadjusted.csv


In [3]:
"""What the FY2027 file is, before anything is computed from it."""
print(f"rows                 {len(fy27):>6}   one per county or New England town")
print(f"distinct fips        {fy27.fips.nunique():>6}")
print(f"distinct FMR areas   {fy27.hud_area_code.nunique():>6}   <- the unit the number is set at")
print(f"metropolitan rows    {int((fy27.metro == 1).sum()):>6}")
print(f"nonmetropolitan rows {int((fy27.metro == 0).sum()):>6}")

per_area = fy27.groupby("hud_area_code").size()
multi = per_area[per_area > 1]
print(
    f"\nFMR areas spanning more than one row: {len(multi)} of {len(per_area)}"
    f"\nrows inside such an area:             {int(multi.sum())} of {len(fy27)}"
    f" ({100 * multi.sum() / len(fy27):.1f}%)"
)
widest = fy27[fy27.hud_area_code == per_area.idxmax()]
print(
    f"\nwidest area: {widest.hud_area_name.iloc[0]}"
    f"\n  {len(widest)} rows share one {BEDROOM}-bedroom FMR of ${widest[FMR_COL].iloc[0]:,}"
)

rows                   4764   one per county or New England town
distinct fips          4764
distinct FMR areas     2606   <- the unit the number is set at
metropolitan rows      1976
nonmetropolitan rows   2788

FMR areas spanning more than one row: 281 of 2606
rows inside such an area:             2439 of 4764 (51.2%)

widest area: Boston-Cambridge-Quincy, MA-NH HUD Metro FMR Area
  114 rows share one 2-bedroom FMR of $3,008


## 1. The definition describes the input, not the published number

24 CFR 888.113 calls an FMR an estimate of the **40th percentile** gross rent. A 40th
percentile sits *below* a median. So a reader who takes the definition at face value expects
FMR to come in under the ACS median gross rent for the same unit size and place.

It does not, and the size of the gap is the point.

In [4]:
"""FMR against the ACS median gross rent for the same unit size and county."""
acs, acs_prov = fetch_acs_wide(
    year=ACS_YEAR,
    variables={ACS_RENT: "median_gross_rent_2br", ACS_RENT_MOE: "margin_of_error"},
    geography="county:*",
    within="state:*",
)
acs = acs.rename(columns={ACS_RENT: "acs_rent", ACS_RENT_MOE: "acs_moe"})
# HUD's fips is the 5-digit county code padded to 10 with 99999.
acs["fips"] = acs["state"] + acs["county"] + "99999"

paired = fy27.merge(acs[["fips", "acs_rent", "acs_moe"]], on="fips", how="inner")
paired = paired.dropna(subset=["acs_rent"])

print(
    f"coverage: {len(paired)} of {len(fy27)} FMR rows join an ACS county "
    f"({100 * len(paired) / len(fy27):.1f}%)."
)
print(
    "  The shortfall is structural, not missing data: New England FMR areas are defined at\n"
    "  town level and the ACS county file has no town rows. Every statistic below is over\n"
    f"  the {len(paired)} rows that do join, and does not describe the other "
    f"{len(fy27) - len(paired)}."
)

ratio = paired[FMR_COL] / paired["acs_rent"]
print(
    f"\nFMR / ACS-{ACS_YEAR} median gross rent, {BEDROOM}-bedroom:"
    f"\n  p10 {ratio.quantile(0.10):.3f}   median {ratio.median():.3f}   p90 {ratio.quantile(0.90):.3f}"
    f"\n  FMR below its own county's ACS median: {int((ratio < 1).sum())} of {len(ratio)}"
    f" ({100 * (ratio < 1).mean():.1f}%)"
)

coverage: 3102 of 4764 FMR rows join an ACS county (65.1%).
  The shortfall is structural, not missing data: New England FMR areas are defined at
  town level and the ACS county file has no town rows. Every statistic below is over
  the 3102 rows that do join, and does not describe the other 1662.

FMR / ACS-2023 median gross rent, 2-bedroom:
  p10 1.141   median 1.286   p90 1.626
  FMR below its own county's ACS median: 35 of 3102 (1.1%)


**The median FMR is well above the median it is nominally a low percentile of.** Nothing is
wrong with either number. They refer to different years: the FY2027 FMR is built from an
earlier ACS and then trended forward to the fiscal year in which it binds, while the ACS
2023 5-year estimate is a backward-looking five-year average. The gap is the projection, plus
whatever local survey evidence HUD accepted.

That is the lesson, and it generalises past this source: **a regulatory definition tells you
what went into a number, not what the number is.** Comparing a published administrative
figure to a contemporaneous survey without accounting for the projection is not a finding
about generosity or accuracy; it is a units error.

The dispersion says something further.

In [5]:
"""How uniform is the projection? If it were a single national factor, it would not vary."""
spread = ratio.quantile(0.90) / ratio.quantile(0.10)
print(
    f"ratio p90 / p10 = {spread:.2f}x"
    f"\n  A single national trend factor would put every county at the same ratio."
    f"\n  The 10th and 90th percentile counties differ by {100 * (spread - 1):.0f}%, so the"
    f"\n  step from ACS to FMR is local, and how local is not published as a field."
)

top = paired.nlargest(5, FMR_COL)[["countyname", "stusps", "hud_area_name", FMR_COL, "acs_rent"]]
top = top.assign(ratio=(top[FMR_COL] / top["acs_rent"]).round(2))
print("\nhighest FMRs, with the ACS median each county reports:")
print(top.to_string(index=False))

ratio p90 / p10 = 1.42x
  A single national trend factor would put every county at the same ratio.
  The 10th and 90th percentile counties differ by 42%, so the
  step from ACS to FMR is local, and how local is not published as a field.

highest FMRs, with the ACS median each county reports:
          countyname stusps                        hud_area_name  fmr_2  acs_rent  ratio
   Santa Cruz County     CA       Santa Cruz-Watsonville, CA MSA   4254    2286.0   1.86
        Marin County     CA San Francisco, CA HUD Metro FMR Area   3697    2764.0   1.34
San Francisco County     CA San Francisco, CA HUD Metro FMR Area   3697    2982.0   1.24
    San Mateo County     CA San Francisco, CA HUD Metro FMR Area   3697    3187.0   1.16
         Napa County     CA                         Napa, CA MSA   3375    2234.0   1.51


## 2. The geography is a statute, not a housing market

An FMR area is an OMB Core-Based Statistical Area, or a HUD subdivision of one, or a
nonmetropolitan county. Whatever the counties inside it look like, they get one number.

The obvious test — does the single FMR fall inside the range of its own counties' median
rents — is contaminated by §1: the FMR sits above nearly every county's ACS median because of
the projection, so it would land outside almost any range for a reason that has nothing to do
with aggregation. The statistic below therefore measures the counties **against each other**,
where the common level shift cancels.

In [6]:
"""Dispersion inside multi-county FMR areas, measured without the level shift."""
grouped = paired.groupby("hud_area_code")
area = grouped.agg(
    n=("fips", "size"),
    lo=("acs_rent", "min"),
    hi=("acs_rent", "max"),
    mid=("acs_rent", "median"),
    fmr=(FMR_COL, "first"),
    name=("hud_area_name", "first"),
)
area = area[area.n > 1]
area["spread_pct"] = 100 * (area.hi - area.lo) / area.lo
area["cv_pct"] = 100 * grouped["acs_rent"].std(ddof=0)[area.index] / grouped["acs_rent"].mean()[area.index]

# The contaminated version, reported so it can be dismissed explicitly rather than quietly.
above = int((area.fmr > area.hi).sum())
inside = int(((area.fmr >= area.lo) & (area.fmr <= area.hi)).sum())
below = int((area.fmr < area.lo).sum())
print(
    f"NOT A FINDING. The single FMR vs its counties' [min, max] ACS median:"
    f"\n  above the max {above}   inside {inside}   below the min {below}   (of {len(area)} areas)"
    f"\n  This is section 1's projection showing up again, not evidence about aggregation."
    f"\n  Discarded.\n"
)

print(f"Multi-county FMR areas with ACS coverage: {len(area)}, holding {int(area.n.sum())} county rows.")
print(
    f"  spread (max/min - 1) of county median rent inside one FMR area:"
    f"\n    median {area.spread_pct.median():.1f}%   p90 {area.spread_pct.quantile(0.90):.1f}%"
    f"   max {area.spread_pct.max():.1f}%"
    f"\n  coefficient of variation: median {area.cv_pct.median():.1f}%   p90 {area.cv_pct.quantile(0.90):.1f}%"
)

counties = paired.merge(area[["spread_pct"]], left_on="hud_area_code", right_index=True, how="left")
inside_multi = int(counties.spread_pct.notna().sum())
print(f"\n  county rows inside a multi-county area: {inside_multi} of {len(paired)}")
for t in (25, 50, 100):
    n = int((counties.spread_pct > t).sum())
    print(f"    ...whose area's internal spread exceeds {t:>3}%: {n:>4} ({100 * n / inside_multi:.1f}% of those rows)")

print("\nwidest-spread areas:")
show = area.nlargest(5, "spread_pct")[["name", "n", "lo", "hi", "fmr", "spread_pct"]]
print(show.to_string(index=False, formatters={"spread_pct": "{:.0f}%".format}))

NOT A FINDING. The single FMR vs its counties' [min, max] ACS median:
  above the max 191   inside 17   below the min 0   (of 208 areas)
  This is section 1's projection showing up again, not evidence about aggregation.
  Discarded.

Multi-county FMR areas with ACS coverage: 208, holding 830 county rows.


  spread (max/min - 1) of county median rent inside one FMR area:
    median 29.2%   p90 88.7%   max 213.2%
  coefficient of variation: median 10.5%   p90 22.1%

  county rows inside a multi-county area: 830 of 3102
    ...whose area's internal spread exceeds  25%:  599 (72.2% of those rows)
    ...whose area's internal spread exceeds  50%:  373 (44.9% of those rows)
    ...whose area's internal spread exceeds 100%:  145 (17.5% of those rows)

widest-spread areas:
                                                             name  n    lo     hi  fmr spread_pct
                                  Richmond, VA HUD Metro FMR Area 16 698.0 2186.0 1651       213%
                              St. Louis, MO-IL HUD Metro FMR Area 13 458.0 1296.0 1349       183%
                                    Puerto Rico HUD Nonmetro Area 12 282.0  770.0  517       173%
Nashville-Davidson--Murfreesboro--Franklin, TN HUD Metro FMR Area 10 688.0 1849.0 1798       169%
             Atlanta-Sandy Springs-Roswel

This is MAUP with a statute behind it. In D27 the scale mismatch was a choice the analyst
made and could revisit; here it is a rule the analyst cannot change, and the payment standard
in the cheapest county of an area is set by a distribution it barely belongs to.

The corpus's usual remedy — go finer — exists here as **Small Area FMRs**, which set the
number at ZIP level in some metropolitan areas. That is the subject of exercise 2, not of
this notebook: SAFMRs apply in a designated subset of metros, so switching to them changes
which places are covered as well as the scale, and the two effects would have to be
separated.

## 3. The machine-readable copy is not the operative schedule

HUD publishes FMRs on its open-data portal as an ArcGIS feature layer — the copy a developer
would actually build against. It carries FMR values by bedroom size and an area code.

It carries **no fiscal-year field**. So the first question is not how to use it but what it is.

In [7]:
"""Query HUD's open-data layer and identify which schedule it holds."""
def arcgis_all(url: str, fields: str, page: int = 1000) -> pd.DataFrame:
    """Page through an ArcGIS feature service, honouring its transfer limit."""
    rows, offset = [], 0
    while True:
        q = (
            f"{url}?where=1%3D1&outFields={fields}&returnGeometry=false"
            f"&resultRecordCount={page}&resultOffset={offset}&f=json"
        )
        with urllib.request.urlopen(q, timeout=90) as response:
            payload = json.loads(response.read())
        if "error" in payload:
            raise RuntimeError(f"ArcGIS returned an error object: {payload['error']}")
        batch = payload.get("features", [])
        rows.extend(f["attributes"] for f in batch)
        if not batch or not payload.get("exceededTransferLimit"):
            break
        offset += len(batch)
    return pd.DataFrame(rows)


live = arcgis_all(ARCGIS, "FMR_CODE,FMR_AREANAME,FMR_0BDR,FMR_1BDR,FMR_2BDR,FMR_3BDR,FMR_4BDR")
live = live.rename(columns={"FMR_CODE": "hud_area_code"}).set_index("hud_area_code")
LIVE_COLS = [f"FMR_{b}BDR" for b in range(5)]

print(f"ArcGIS layer:    {len(live):>5} rows, {live.index.nunique()} distinct area codes")
print(f"FY{FISCAL_YEAR} schedule: {fy27.hud_area_code.nunique():>5} areas")
print(
    "\n  The reduced FY2026 file carries no area column, so its own area count is not\n"
    "  available here; it is not inferred from the FY2027 codes, which would only\n"
    "  restate FY2027."
)
orphans = live.index.difference(fy27.hud_area_code.unique())
print(
    f"\n  area codes in the layer with no counterpart in FY{FISCAL_YEAR}: {len(orphans)}"
    f"\n  -> the layer is keyed to a geography that FY{FISCAL_YEAR} no longer uses, which is the"
    f"\n     first sign that it is not the current schedule."
)

ArcGIS layer:     2622 rows, 2622 distinct area codes
FY2027 schedule:  2606 areas

  The reduced FY2026 file carries no area column, so its own area count is not
  available here; it is not inferred from the FY2027 codes, which would only
  restate FY2027.

  area codes in the layer with no counterpart in FY2027: 16
  -> the layer is keyed to a geography that FY2027 no longer uses, which is the
     first sign that it is not the current schedule.


In [8]:
"""Compare the layer against each published schedule, on all five bedroom sizes."""
def by_area(frame: pd.DataFrame) -> pd.DataFrame:
    """Collapse a county-level schedule to one row per FMR area."""
    if "hud_area_code" not in frame.columns:
        frame = frame.merge(fy27[["fips", "hud_area_code"]], on="fips")
    return frame.groupby("hud_area_code")[FMR_COLS].first()


def agreement(schedule: pd.DataFrame, label: str) -> tuple[int, int]:
    joined = live.join(by_area(schedule), how="inner")
    same = np.logical_and.reduce(
        [joined[a].to_numpy() == joined[b].to_numpy() for a, b in zip(LIVE_COLS, FMR_COLS)]
    )
    print(
        f"  ArcGIS vs {label:<22} identical on all five bedroom sizes in "
        f"{int(same.sum()):>5} of {len(joined):>5} areas ({100 * same.mean():5.1f}%)"
    )
    return int(same.sum()), len(joined)

print("Which schedule is the open-data layer?")
n27, d27 = agreement(fy27, f"FY{FISCAL_YEAR}")
n26, d26 = agreement(fy26, "FY2026 as published")
n26r, d26r = agreement(fy26r, "FY2026 revised")

Which schedule is the open-data layer?
  ArcGIS vs FY2027                 identical on all five bedroom sizes in     0 of  2606 areas (  0.0%)
  ArcGIS vs FY2026 as published    identical on all five bedroom sizes in  2604 of  2606 areas ( 99.9%)
  ArcGIS vs FY2026 revised         identical on all five bedroom sizes in  2597 of  2606 areas ( 99.7%)


In [9]:
"""Pin it down: the layer's treatment of the areas FY2026 later revised."""
orig, rev = by_area(fy26), by_area(fy26r)
revised_areas = orig.index[(orig != rev).any(axis=1)]

joined = live.join(rev, how="inner")
matches_revised = np.logical_and.reduce(
    [joined[a].to_numpy() == joined[b].to_numpy() for a, b in zip(LIVE_COLS, FMR_COLS)]
)
on_revised = int(pd.Series(matches_revised, index=joined.index).loc[
    joined.index.intersection(revised_areas)
].sum())
print(
    f"FMR areas revised during FY2026: {len(revised_areas)}"
    f"\n  the layer carries the REVISED values for {on_revised} of them"
    f"\n  -> the layer is the pre-revision FY2026 schedule."
)

unmatched = live.join(by_area(fy26), how="inner")
neither = ~np.logical_and.reduce(
    [unmatched[a].to_numpy() == unmatched[b].to_numpy() for a, b in zip(LIVE_COLS, FMR_COLS)]
)
odd = unmatched[neither]
print(f"\nAreas where the layer matches neither FY2026 publication: {len(odd)}")
if len(odd):
    print(odd[["FMR_AREANAME"] + LIVE_COLS + FMR_COLS].to_string())

FMR areas revised during FY2026: 7
  the layer carries the REVISED values for 0 of them
  -> the layer is the pre-revision FY2026 schedule.

Areas where the layer matches neither FY2026 publication: 2
                                   FMR_AREANAME  FMR_0BDR  FMR_1BDR  FMR_2BDR  FMR_3BDR  FMR_4BDR  fmr_0  fmr_1  fmr_2  fmr_3  fmr_4
hud_area_code                                                                                                                       
METRO47930M47930      Waterbury-Shelton, CT MSA      1246      1445      1788      2210      2608   1246   1445   1788   2210   2864
NCNTY72923N72923  Puerto Rico HUD Nonmetro Area       414       433       475       661       717    461    479    531    743    805


So the machine-readable distribution — the one whose entire purpose is programmatic use — is
a fiscal year behind, sits on the wrong side of a revision, does not exactly reproduce any
published schedule, and says none of this anywhere in its payload. Its own metadata describes
what an FMR is for at length and never states which year it holds.

There is no sentinel to catch this and no flag to check. The only detection available is the
one performed above: hold the machine-readable copy against the authoritative publication and
count. **Add it to the handbook's sentinel catalogue as the case with no sentinel** — a source
that is not wrong in any cell, and is wrong as a whole.

## 4. A revision with no revision series

D08 established that FRED series get revised, and that ALFRED lets you ask for any vintage.
FMRs are revised too — HUD grants reevaluation requests after a schedule takes effect — but
there is no vintage parameter and no revision series. There are two files with similar names,
and the difference between them is the revision.

In [10]:
"""Difference the two FY2026 publications."""
pair = fy26.merge(fy26r, on="fips", suffixes=("_orig", "_rev"))
changed = np.logical_or.reduce(
    [pair[f"{c}_orig"].to_numpy() != pair[f"{c}_rev"].to_numpy() for c in FMR_COLS]
)
diff = pair[changed].merge(
    fy27[["fips", "countyname", "stusps", "hud_area_name", "hud_area_code"]], on="fips"
)
print(
    f"county rows whose FY2026 FMR changed after publication: {len(diff)} of {len(pair)}"
    f"\nFMR areas affected: {diff.hud_area_code.nunique()}\n"
)
diff = diff.assign(pct=100 * (diff[f"{FMR_COL}_rev"] / diff[f"{FMR_COL}_orig"] - 1))
print(
    diff.sort_values("pct", ascending=False)[
        ["countyname", "stusps", "hud_area_name", f"{FMR_COL}_orig", f"{FMR_COL}_rev", "pct"]
    ].to_string(index=False, formatters={"pct": "{:+.1f}%".format})
)
print(
    f"\nrange of the change, {BEDROOM}-bedroom: {diff.pct.min():+.1f}% to {diff.pct.max():+.1f}%"
    f"\nevery change raised the number: {bool((diff.pct > 0).all())}"
)

county rows whose FY2026 FMR changed after publication: 9 of 4764
FMR areas affected: 7

            countyname stusps                                          hud_area_name  fmr_2_orig  fmr_2_rev    pct
           Napa County     CA                                           Napa, CA MSA        2773       3315 +19.5%
       Buncombe County     NC                                      Asheville, NC MSA        1567       1835 +17.1%
      Henderson County     NC                                      Asheville, NC MSA        1567       1835 +17.1%
        Madison County     NC                                      Asheville, NC MSA        1567       1835 +17.1%
   Transylvania County     NC                                Transylvania County, NC        1164       1363 +17.1%
           Linn County     OR                                         Albany, OR MSA        1500       1695 +13.0%
         Benton County     OR                                      Corvallis, OR MSA        1622       182

Nine county rows, seven FMR areas, every one an increase. That direction is not an accident of
the sample: a reevaluation is something a housing authority *requests*, and an authority
requests one when the published FMR is too low to lease units. The revision series, if it
existed, would be systematically upward — which is a property of the appeals process, not of
the rental market, and would bias any study that treated FMR revisions as new information
about rents.

**What this costs a reader.** Between the two files there is nothing to distinguish them but
a filename. An analysis built on `FY26_FMRs.xlsx` is not wrong about anything it can see; it
is simply using the superseded number for Los Angeles County, where the FY2026 2-bedroom
payment standard was ultimately 11.6% higher than the file says.

## 4b. Two machine-readable distributions, and only one is current

The layer in §3 is not HUD's only programmatic surface. There is also a keyed REST API, and
it is worth asking the same question of it — because a publisher shipping two machine-readable
copies of a binding number is shipping a choice, whether or not it says so.

This section also does something the corpus asks for everywhere and rarely gets: it validates
the pinned bulk files against an **independent retrieval path from the same publisher**.

In [11]:
"""Re-fetch a sample of counties from HUD's API and compare with the pinned files.

The API is rate limited to 60 requests a minute, so this samples rather than sweeps.
`fetch_hud_fmr_county` requires an explicit year: HUD defaults to the current schedule,
and an un-pinned call would silently change answers each October.
"""
from krl_tutorials_data.sources import ResponseShapeError, fetch_hud_fmr_county

SAMPLE_SIZE = 20
sample = fy27.fips.sort_values().iloc[:: max(1, len(fy27) // SAMPLE_SIZE)].head(SAMPLE_SIZE)

agree, differ, no_county_figure = 0, [], 0
for entity in sample:
    try:
        live_row, _ = fetch_hud_fmr_county(fips=entity, year=FISCAL_YEAR)
    except ResponseShapeError:
        # A Small Area FMR metro has no single county figure; see below.
        no_county_figure += 1
        continue
    theirs = [int(live_row[c].iloc[0]) for c in FMR_COLS]
    ours = [int(fy27.loc[fy27.fips == entity, c].iloc[0]) for c in FMR_COLS]
    if theirs == ours:
        agree += 1
    else:
        differ.append((entity, theirs, ours))

print(
    f"API vs the pinned FY{FISCAL_YEAR} file, {len(sample)} counties sampled:"
    f"\n  identical on all five bedroom sizes: {agree}"
    f"\n  different:                           {len(differ)}"
    f"\n  no single county figure returned:    {no_county_figure}"
)
for entity, theirs, ours in differ:
    print(f"    {entity}  api={theirs}  file={ours}")

API vs the pinned FY2027 file, 20 counties sampled:
  identical on all five bedroom sizes: 16
  different:                           0
  no single county figure returned:    4


In [12]:
"""And the question §3 asked of the layer, asked of the API."""
probe = diff.fips.tolist()          # the nine rows FY2026 revised
served_revised, served_original, served_neither, no_figure = 0, 0, 0, 0
for entity in probe:
    try:
        live_row, _ = fetch_hud_fmr_county(fips=entity, year=2026)
    except ResponseShapeError:
        no_figure += 1
        continue
    theirs = [int(live_row[c].iloc[0]) for c in FMR_COLS]
    original = [int(diff.loc[diff.fips == entity, f"{c}_orig"].iloc[0]) for c in FMR_COLS]
    revised = [int(diff.loc[diff.fips == entity, f"{c}_rev"].iloc[0]) for c in FMR_COLS]
    if theirs == revised:
        served_revised += 1
    elif theirs == original:
        served_original += 1
    else:
        served_neither += 1

print(
    f"For the {len(probe)} county rows FY2026 revised, asking HUD's API for FY2026:"
    f"\n  it serves the REVISED value:  {served_revised}"
    f"\n  it serves the ORIGINAL value: {served_original}"
    f"\n  neither:                      {served_neither}"
    f"\n  no single county figure:      {no_figure}"
)
print(
    "\nSo HUD publishes two machine-readable copies of the same binding number and they"
    "\ndisagree: the API is current and post-revision, the feature layer is a fiscal year"
    "\nbehind and pre-revision. Neither states which it is. A developer's answer depends"
    "\nentirely on which surface they happened to find."
)

For the 9 county rows FY2026 revised, asking HUD's API for FY2026:
  it serves the REVISED value:  8
  it serves the ORIGINAL value: 0
  neither:                      0
  no single county figure:      1

So HUD publishes two machine-readable copies of the same binding number and they
disagree: the API is current and post-revision, the feature layer is a fiscal year
behind and pre-revision. Neither states which it is. A developer's answer depends
entirely on which surface they happened to find.


**One more thing the API makes visible.** For a metropolitan area that sets Small Area FMRs,
the API does not return a county figure at all — it returns a list of ZIP-level records. Los
Angeles County returns 475 of them. `fetch_hud_fmr_county` raises rather than averaging,
because a county-level FMR for such an area is not a number HUD publishes, and computing one
would manufacture a payment standard that does not exist. The bulk file, by contrast, prints
a single county row for the same place — so the two surfaces disagree about whether the
quantity exists, not merely about its value.

## 5. Year over year, the binding number does not move as one thing

In [13]:
"""FY2026 -> FY2027, by bedroom size."""
yoy = fy27.merge(fy26, on="fips", suffixes=("_27", "_26"))
print(f"county rows present in both schedules: {len(yoy)} of {len(fy27)} (FY2027)\n")
rows = []
for b in range(5):
    delta = yoy[f"fmr_{b}_27"].to_numpy() - yoy[f"fmr_{b}_26"].to_numpy()
    pct = 100 * (yoy[f"fmr_{b}_27"] / yoy[f"fmr_{b}_26"] - 1)
    rows.append(
        {"bedrooms": b, "fell": int((delta < 0).sum()), "unchanged": int((delta == 0).sum()),
         "rose": int((delta > 0).sum()), "median_pct": pct.median()}
    )
table = pd.DataFrame(rows)
print(table.to_string(index=False, formatters={"median_pct": "{:+.2f}%".format}))

signs = np.sign(
    yoy[[f"fmr_{b}_27" for b in range(5)]].to_numpy()
    - yoy[[f"fmr_{b}_26" for b in range(5)]].to_numpy()
)
mixed = int(((signs > 0).any(axis=1) & (signs < 0).any(axis=1)).sum())
print(
    f"\ncounties where the binding number ROSE for one bedroom size and FELL for another,"
    f"\nin the same place in the same year: {mixed} of {len(yoy)} ({100 * mixed / len(yoy):.1f}%)"
)
fell = int((yoy[f"fmr_{BEDROOM}_27"] < yoy[f"fmr_{BEDROOM}_26"]).sum())
print(
    f"\n{BEDROOM}-bedroom payment standard fell in {fell} of {len(yoy)} counties"
    f" ({100 * fell / len(yoy):.1f}%), against a median change of"
    f" {100 * (yoy[f'fmr_{BEDROOM}_27'] / yoy[f'fmr_{BEDROOM}_26'] - 1).median():+.2f}%."
)

county rows present in both schedules: 4758 of 4764 (FY2027)

 bedrooms  fell  unchanged  rose median_pct
        0  1247          7  3504     +3.78%
        1  1363         11  3384     +3.73%
        2  1338          4  3416     +3.47%
        3  1407         35  3316     +3.00%
        4  1435          5  3318     +2.82%

counties where the binding number ROSE for one bedroom size and FELL for another,
in the same place in the same year: 1109 of 4758 (23.3%)

2-bedroom payment standard fell in 1338 of 4758 counties (28.1%), against a median change of +3.47%.


A headline of "+3.5%" is true and describes almost nobody: a quarter of counties saw the
2-bedroom standard fall, and in a further large group the standard moved in *both* directions
at once depending on unit size. Averaging across bedroom sizes — the natural thing to do when
summarising — destroys precisely the structure that determines who is affected.

## 6. A decomposition this source does not support

HUD also publishes an "unadjusted rents" file. It is tempting: the ratio of the final FMR to
the unadjusted rent looks like the policy adjustment, isolated, which would let us decompose
the §1 gap into base and adjustment.

Before using it that way, check what it covers.

In [14]:
"""Establish the unadjusted file's coverage before relying on it for anything."""
covered = set(unadj.fips)
labelled = fy27.assign(is_covered=fy27.fips.isin(covered))
area_cov = labelled.groupby("hud_area_code").agg(
    metro=("metro", "max"), n=("fips", "size"), covered=("is_covered", "max")
)
print(
    f"unadjusted file: {len(unadj)} rows, covering "
    f"{int(area_cov.covered.sum())} of {len(area_cov)} FMR areas "
    f"({100 * area_cov.covered.mean():.1f}%)\n"
)
print("coverage by area type:")
print(area_cov.groupby("metro")["covered"].agg(areas="size", covered="sum")
      .rename(index={0: "nonmetropolitan", 1: "metropolitan"}).to_string())
print("\ncoverage by whether the area spans more than one row:")
print(area_cov.assign(multi=area_cov.n > 1).groupby("multi")["covered"]
      .agg(areas="size", covered="sum")
      .rename(index={False: "single-row area", True: "multi-row area"}).to_string())

joined = fy27.merge(unadj[["fips", f"unadj_br{BEDROOM}"]], on="fips", how="inner")
r = joined[FMR_COL] / joined[f"unadj_br{BEDROOM}"]
print(
    f"\nWhere both exist ({len(joined)} rows), FMR / unadjusted:"
    f" median {r.median():.3f}, max {r.max():.3f}, exactly 1.000 in {int((r == 1).sum())}."
)
print(
    "\nREFUSED. That ratio is not reported as the policy adjustment. The file covers"
    f"\n{int(area_cov[area_cov.metro == 0].covered.sum())} of {int((area_cov.metro == 0).sum())}"
    " nonmetropolitan areas but only"
    f" {int(area_cov[area_cov.metro == 1].covered.sum())} of {int((area_cov.metro == 1).sum())}"
    " metropolitan ones,"
    f"\nand {int(area_cov[area_cov.n > 1].covered.sum())} of {int((area_cov.n > 1).sum())}"
    " multi-row areas -- it is overwhelmingly small, single-county,"
    "\nnonmetropolitan places. HUD also states the file is for setting public housing flat"
    "\nrents, not for reconstructing an FMR. A ratio computed on this subset would describe"
    "\nthe adjustment in small rural areas and be presented as the national one."
)

unadjusted file: 1057 rows, covering 1048 of 2606 FMR areas (40.2%)

coverage by area type:
                 areas  covered
metro                          
nonmetropolitan   1957      989
metropolitan       649       59

coverage by whether the area spans more than one row:
                 areas  covered
multi                          
single-row area   2325     1040
multi-row area     281        8

Where both exist (1057 rows), FMR / unadjusted: median 1.090, max 1.905, exactly 1.000 in 0.

REFUSED. That ratio is not reported as the policy adjustment. The file covers
989 of 1957 nonmetropolitan areas but only 59 of 649 metropolitan ones,
and 8 of 281 multi-row areas -- it is overwhelmingly small, single-county,
nonmetropolitan places. HUD also states the file is for setting public housing flat
rents, not for reconstructing an FMR. A ratio computed on this subset would describe
the adjustment in small rural areas and be presented as the national one.


This is the shortest section and the one most worth copying. The file was available, the join
worked, the ratio was plausible, and every number it produced would have been real. The
finding is that its **selection** makes it unfit for the claim, and no property of the
computed ratio would have revealed that. The check cost three lines; skipping it would have
produced a confident national statistic about rural counties.

In [15]:
"""FINDINGS"""
print("FINDINGS")
print("=" * 78)
print(f"""
1. FMR is defined as a 40th-percentile gross rent and publishes at a median
   {ratio.median():.3f}x the ACS {ACS_YEAR} median gross rent for the same unit size and county;
   it falls below that median in only {int((ratio < 1).sum())} of {len(ratio)} counties. The definition
   describes the input. The gap is the projection to the fiscal year, and the
   ratio's p90/p10 of {ratio.quantile(0.90) / ratio.quantile(0.10):.2f}x shows the projection is local, not national.
2. One number binds many markets. {len(multi)} of {len(per_area)} FMR areas span more than one
   county row, covering {int(multi.sum())} of {len(fy27)} rows; {len(widest)} rows share the single
   {BEDROOM}-bedroom FMR of ${widest[FMR_COL].iloc[0]:,} in {widest.hud_area_name.iloc[0]}.
   Measured against each other so the projection cancels, county median rents
   inside one FMR area span a median of {area.spread_pct.median():.1f}% (p90 {area.spread_pct.quantile(0.90):.1f}%, max {area.spread_pct.max():.1f}%),
   and {int((counties.spread_pct > 25).sum())} of {inside_multi} such rows sit in an area whose internal spread exceeds 25%.
3. HUD's own machine-readable layer is not the operative schedule. It matches
   FY{FISCAL_YEAR} in {n27} of {d27} areas and FY2026-as-published in {n26} of {d26}; for the
   {len(revised_areas)} areas FY2026 later revised it carries the revised values in {on_revised}.
   It is the pre-revision FY2026 schedule, it matches neither publication in {len(odd)} areas,
   and it carries no fiscal-year field. No sentinel detects this.
4. FY2026 was revised after it took effect: {len(diff)} county rows across
   {diff.hud_area_code.nunique()} FMR areas, {diff.pct.min():+.1f}% to {diff.pct.max():+.1f}%, every one an increase.
   HUD publishes no revision series; the change exists only as the difference
   between two files, and the upward direction is a property of an appeals
   process rather than of the rental market.
5. The annual change is not one thing. {fell} of {len(yoy)} counties ({100 * fell / len(yoy):.1f}%) saw the
   {BEDROOM}-bedroom standard FALL against a median of {100 * (yoy[f'fmr_{BEDROOM}_27'] / yoy[f'fmr_{BEDROOM}_26'] - 1).median():+.2f}%, and in {mixed}
   counties ({100 * mixed / len(yoy):.1f}%) it rose for one bedroom size and fell for another.
6. The unadjusted-rents file covers {int(area_cov.covered.sum())} of {len(area_cov)} areas and is
   {int(area_cov[area_cov.metro == 0].covered.sum())}/{int((area_cov.metro == 0).sum())} nonmetropolitan against {int(area_cov[area_cov.metro == 1].covered.sum())}/{int((area_cov.metro == 1).sum())} metropolitan. The
   FMR-to-unadjusted ratio is therefore refused as a national decomposition.
7. HUD ships two machine-readable copies of the same binding number and they
   disagree. Sampling {len(sample)} counties, the API matches the pinned FY{FISCAL_YEAR} file in
   {agree} and differs in {len(differ)}; asked for FY2026 on the {len(probe)} revised rows it serves
   the REVISED value in {served_revised} and the original in {served_original}. The feature layer
   serves the pre-revision FY2026 schedule for the same rows. Neither surface
   states which schedule it holds.
8. Coverage: {len(paired)} of {len(fy27)} FMR rows ({100 * len(paired) / len(fy27):.1f}%) join an ACS county. The
   remainder are New England town-level areas the ACS county file does not carry.
""".rstrip())
print("=" * 78)

FINDINGS

1. FMR is defined as a 40th-percentile gross rent and publishes at a median
   1.286x the ACS 2023 median gross rent for the same unit size and county;
   it falls below that median in only 35 of 3102 counties. The definition
   describes the input. The gap is the projection to the fiscal year, and the
   ratio's p90/p10 of 1.42x shows the projection is local, not national.
2. One number binds many markets. 281 of 2606 FMR areas span more than one
   county row, covering 2439 of 4764 rows; 114 rows share the single
   2-bedroom FMR of $3,008 in Boston-Cambridge-Quincy, MA-NH HUD Metro FMR Area.
   Measured against each other so the projection cancels, county median rents
   inside one FMR area span a median of 29.2% (p90 88.7%, max 213.2%),
   and 599 of 830 such rows sit in an area whose internal spread exceeds 25%.
3. HUD's own machine-readable layer is not the operative schedule. It matches
   FY2027 in 0 of 2606 areas and FY2026-as-published in 2604 of 2606; for the
   7 

## Limitations

- **One bedroom size.** Everything is reported for 2-bedroom units. §5 shows bedroom sizes
  move differently, so the §1 and §2 magnitudes should not be assumed to hold at 0 or 4.
- **65% ACS coverage.** New England FMR areas are town-level and drop out of every
  FMR-versus-ACS statistic. Those areas are disproportionately in a few states, so the
  coverage loss is not geographically neutral.
- **ACS margins are carried but not propagated.** `acs_moe` is retrieved and unused. The §2
  dispersion statistics are computed on point estimates, and a county whose median rent has a
  wide margin contributes to the spread as though it were known. Exercise 4.
- **The projection is inferred, not decomposed.** §1 attributes the FMR-to-ACS gap to
  trending forward plus local survey evidence. That is HUD's documented method, not something
  this notebook measures — §6 is the attempt to measure it, and its refusal.
- **The API cross-check is a sample, not a sweep.** §4b re-fetches 20 counties under HUD's
  60-per-minute limit. It establishes that the pinned files agree with HUD on those, not that
  all 4,764 rows do.
- **One fiscal year of revision.** §4 observes FY2026. Whether nine rows is typical is not
  established here.
- **Not an evaluation of the program.** Nothing here says whether FMRs are set well. The
  subject is what kind of number this is and how to handle it.

## Key Takeaways

- **A regulatory definition describes what went into a number, not what the number is.**
  Check the gap; here it is a median 28.6% and it is not uniform.
- **When the geography is set by statute, the scale mismatch is not a modelling choice.**
  Measure what the single number is required to ignore — and measure it in a way the common
  level shift cannot fake, or you will report the projection twice.
- **Report the contaminated statistic and discard it in the open.** §2 does this deliberately:
  a reader who would have computed it needs to see why it does not answer the question.
- **A machine-readable distribution is a claim, not a fact.** Hold it against the
  authoritative publication and count. Nothing in the payload will tell you — and when a
  publisher ships two, as HUD does, one of them can be a year out of date.
- **Read the headers, not the status code.** HTTP 202 with an empty body is a bot
  challenge that passes every `status_code < 400` check ever written. It was misread as
  an egress block during this build until `x-amzn-waf-action` was actually looked at.
- **A revision without a revision series is still a revision.** Two files with similar names
  are a vintage system with no interface.
- **Check a file's coverage before you check its values.** A ratio computed on a selected
  subset is real in every cell and wrong as an answer.

## Exercises

1. **The other bedroom sizes.** Repeat §1 and §2 at 0 and 4 bedrooms, using `B25031_002E`
   and `B25031_006E`. Does the FMR-to-ACS ratio depend on unit size, and does the ranking of
   areas by internal spread survive the change?
2. **Small Area FMRs.** `FY27_safmrs.xlsx` sets the number at ZIP level in designated metros.
   Acquire it, and quantify how much of §2's within-area dispersion SAFMRs actually resolve —
   separating the change of scale from the change in which places are covered.
3. **Who bears the aggregation.** Within each multi-county area, rank counties by median rent
   and compute the gap between the single FMR and a county-specific 40th percentile. Is the
   burden systematically on the cheaper counties, the more populous ones, or neither?
4. **Propagate the margins.** Redo §2 carrying `acs_moe`. Under the independence and the
   perfect-correlation bounds from `standardization.py`, how many areas still have a spread
   distinguishable from zero?
5. **Is nine typical?** FY2025 and earlier schedules are published the same way. Diff each
   year's original against its revision and establish whether FY2026's nine rows is a normal
   year, and whether the same areas recur.
6. **Catch the challenge, elsewhere.** `reject_bot_challenge` now runs after every request in
   `sources.py`, and `tests/test_fetcher_response_contract.py` records what the sweep found:
   no fetcher ever *accepted* a challenge, but only one named it correctly. Take the same
   question to a source this corpus does not yet use — pick one, inject a 202 with an empty
   body, and see whether its client library reports the cause or a parse failure.
7. **Write the rule.** Draft the paragraph this corpus should use for administratively
   determined numbers: when such a number may be compared to a survey estimate, what must be
   stated when it is, and what a notebook must record about which publication it used. Then
   check §5.4 of the Data Source Handbook against it.

## References

1. **24 CFR 888.113** — *Fair market rent schedule for existing housing.*
   The definition quoted at the top.
2. **42 USC 1437f(c)(1)** — the publication and effective-date requirement.
3. **U.S. Department of Housing and Urban Development.** *Fair Market Rents.*
   <https://www.huduser.gov/portal/datasets/fmr.html>
4. **U.S. Department of Housing and Urban Development.** *FY2027 Fair Market Rent
   Documentation System* and *FY27 Public FMR Methodology.* The trend and local-survey
   procedure that §1's gap reflects.
5. **U.S. Census Bureau.** *American Community Survey 5-Year Estimates, 2023*, table B25031
   (median gross rent by bedrooms).
6. **Openshaw, S.** (1984). *The Modifiable Areal Unit Problem.* Geo Books. — §2, with the
   difference that here the areal units are fixed by regulation.
7. **Data Source Handbook**, `docs/methodology.md` — §1 for the five regimes
   this source does not fit, §5.3 for bulk-file pinning.